# Pipeline 5 - Water Fraction threshold

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
OUTPUT_DIR = "/data/swot/output"

## Read the sites and dates

In [7]:
import numpy as np
import pandas as pd

from swot_toolkit.analysis import open_sites_and_dates
from swot_toolkit.metrics import calc_metrics, process_swot_mask
from swot_toolkit.pipe4 import create_swot_mosaic, open_datasets, quality_flags_bad

sites_dates = open_sites_and_dates("/data/swot/output")

METRICS = ["iou", "f1", "precision", "recall"]

In [22]:
metrics = ["iou", "f1", "precision", "recall"]

overall_results = pd.DataFrame()
for site, dates in sites_dates.items():
    for date in dates:
        # Load the datasets
        datasets = open_datasets(site, date)

        # Now, let's create the Raster 100 array
        swot_mask, _, _ = create_swot_mosaic(
            region_name=site,
            ref_date=date,
            exclude_flags=quality_flags_bad,
            exclude_no_data=True,
        )

        # Once the datasets are open, we can apply the threshold
        results = {}
        for thresh in np.arange(0, 1.0, 0.05):
            mask = process_swot_mask(swot_mask, water_threshold=thresh)

            title = f"water >= {thresh:.2f}"
            results[title] = calc_metrics(
                datasets["ref_mask"],
                mask,
                metrics,
                binary=True,
            ).rename(columns={0: title})

        metrics_df = pd.concat(results.values(), axis=1)
        metrics_df.index = pd.MultiIndex.from_product([[f"{site} {date}"], metrics_df.index])
        overall_results = pd.concat([overall_results, metrics_df], axis=0)


In [25]:
site_means = overall_results.groupby(level=1).mean()
site_means.index = pd.MultiIndex.from_product([["MEAN"], site_means.index])

site_medians = overall_results.groupby(level=1).median()
site_medians.index = pd.MultiIndex.from_product([["MEDIAN"], site_medians.index])

overall_results = pd.concat([overall_results, site_means, site_medians], axis=0)


In [43]:
columns = site_means.columns[::2]
overall_results[columns]

water >= 0.00  water >= 0.10  water >= 0.20  \
Curua-Una 2025-08-14   iou             0.028900       0.641600       0.711800   
                       f1              0.056100       0.781700       0.831600   
                       precision       0.028900       0.662000       0.750300   
                       recall          0.999900       0.954200       0.932700   
Northeast 2025-07-20   iou             0.017900       0.439400       0.577400   
                       f1              0.035200       0.610600       0.732100   
                       precision       0.017900       0.441400       0.585000   
                       recall          0.999800       0.989900       0.978100   
Rio_Branco 2025-09-07  iou             0.055600       0.384100       0.401900   
                       f1              0.105300       0.555000       0.573300   
                       precision       0.055600       0.413400       0.444900   
                       recall          0.995200       0.844100       0.806100   
Rio_Madeira 2025-07-21 iou             0.059800       0.552600       0.600200   
                       f1              0.112800       0.711900       0.750200   
                       precision       0.059800       0.561200       0.614500   
                       recall          0.999600       0.973100       0.962800   
Rio_Negro 2025-08-07   iou             0.262000       0.709700       0.737100   
                       f1              0.415200       0.830200       0.848600   
                       precision       0.262000       0.721100       0.757400   
                       recall          0.999400       0.978300       0.964900   
MEAN                   f1              0.144920       0.697880       0.747160   
                       iou             0.084840       0.545480       0.605680   
                       precision       0.084840       0.559820       0.630420   
                       recall          0.998780       0.947920       0.928920   
MEDIAN                 f1              0.105300       0.711900       0.750200   
                       iou             0.055600       0.552600       0.600200   
                       precision       0.055600       0.561200       0.614500   
                       recall          0.999600       0.973100       0.962800   
MEAN                   f1              0.139260       0.699883       0.747594   
                       iou             0.080663       0.546497       0.604897   
                       precision       0.080663       0.560017       0.628146   
                       recall          0.998897       0.951517       0.933760   
MEDIAN                 f1              0.105300       0.711900       0.750200   
                       iou             0.055600       0.552600       0.600200   
                       precision       0.055600       0.561200       0.614500   
                       recall          0.999600       0.973100       0.962800   

                                  water >= 0.30  water >= 0.40  water >= 0.50  \
Curua-Una 2025-08-14   iou             0.745500       0.759200       0.760200   
                       f1              0.854200       0.863100       0.863800   
                       precision       0.804000       0.842000       0.873200   
                       recall          0.911100       0.885300       0.854500   
Northeast 2025-07-20   iou             0.667400       0.727100       0.756400   
                       f1              0.800500       0.842000       0.861300   
                       precision       0.687300       0.770600       0.834800   
                       recall          0.958300       0.928000       0.889600   
Rio_Branco 2025-09-07  iou             0.411300       0.425200       0.476500   
                       f1              0.582900       0.596700       0.645400   
                       precision       0.469200       0.504900       0.616500   
                       recall          0.769400       0.729400  

In [27]:
import plotly.graph_objects as go


In [28]:
styles = {
    "precision": {"line": {"width": 1, "color": "blue", "dash": "dash"}},
    "recall": {"line": {"width": 1, "color": "green", "dash": "dash"}},
    "f1": {"line": {"width": 3, "color": "purple", "dash": "solid"}},
    "iou": {"line": {"width": 3, "color": "red", "dash": "solid"}},
}

In [29]:
site_means.index.get_level_values(1)

Index(['f1', 'iou', 'precision', 'recall'], dtype='object')

### MEAN graph

In [30]:
fig = go.Figure()

for idx, row in site_means.iterrows():
    x = [float(c[-4:]) for c in site_means.columns]
    name = idx[1] if isinstance(idx, tuple) else idx
    scatter = go.Scatter(x=x, y=row, mode="lines", name=name, line=styles[name]["line"])
    fig.add_trace(scatter)

# Add axis labels
fig.update_layout(xaxis_title="Water Fraction Threshold", yaxis_title="Values")


loc = ("MEAN", "f1") if isinstance(idx, tuple) else "f1"
# Find F1 value at x=0.55 and add annotation
f1_at_055 = site_means.loc[loc, "water >= 0.60"]
fig.add_annotation(
    x=0.55,
    y=f1_at_055,
    text=f"max(F1): x=0.55, y={f1_at_055:.3f}",
    showarrow=True,
    arrowhead=2,
    arrowsize=1,
    arrowwidth=2,
    arrowcolor="purple",
    bgcolor="white",
    bordercolor="purple",
    borderwidth=1,
)

fig

### MEDIAN graph

In [33]:
site_medians[site_medians.columns[::2]]

water >= 0.00  water >= 0.10  water >= 0.20  water >= 0.30  \
MEDIAN f1                0.1053         0.7119         0.7502         0.8005   
       iou               0.0556         0.5526         0.6002         0.6674   
       precision         0.0556         0.5612         0.6145         0.6873   
       recall            0.9996         0.9731         0.9628         0.9488   

                  water >= 0.40  water >= 0.50  water >= 0.60  water >= 0.70  \
MEDIAN f1                0.8420         0.8613         0.8738         0.8551   
       iou               0.7271         0.7564         0.7759         0.7469   
       precision         0.7706         0.8348         0.8979         0.9145   
       recall            0.9280         0.8896         0.8398         0.7841   

                  water >= 0.80  water >= 0.90  
MEDIAN f1                0.8240         0.7646  
       iou               0.7007         0.6189  
       precision         0.9286         0.9392  
       recall            0.7394         0.6387

In [42]:
fig = go.Figure()

for idx, row in site_medians.iterrows():
    x = [float(c[-4:]) for c in site_medians.columns]
    name = idx[1] if isinstance(idx, tuple) else idx
    scatter = go.Scatter(x=x, y=row, mode="lines", name=name, line=styles[name]["line"])
    fig.add_trace(scatter)

# Add axis labels
fig.update_layout(xaxis_title="Water Fraction Threshold", yaxis_title="Values")


loc = ("MEDIAN", "f1") if isinstance(idx, tuple) else "f1"
# Find max F1 value and corresponding x to add annotation
y_max = site_medians.iloc[0].max()
x_y_max = site_medians.iloc[0].idxmax().split(" >= ")[-1]

fig.add_annotation(
    x=x_y_max,
    y=y_max,
    text=f"max(F1): x={x_y_max}, y={y_max:.3f}",
    showarrow=True,
    arrowhead=2,
    arrowsize=1,
    arrowwidth=2,
    arrowcolor="purple",
    bgcolor="white",
    bordercolor="purple",
    borderwidth=1,
)

fig.update_layout(title="Median Metrics across Sites")

### Only IoU

In [ ]:
site_medians[site_medians.columns[::2]]

water >= 0.00  water >= 0.10  water >= 0.20  water >= 0.30  \
MEDIAN f1                0.1053         0.7119         0.7502         0.8005   
       iou               0.0556         0.5526         0.6002         0.6674   
       precision         0.0556         0.5612         0.6145         0.6873   
       recall            0.9996         0.9731         0.9628         0.9488   

                  water >= 0.40  water >= 0.50  water >= 0.60  water >= 0.70  \
MEDIAN f1                0.8420         0.8613         0.8738         0.8551   
       iou               0.7271         0.7564         0.7759         0.7469   
       precision         0.7706         0.8348         0.8979         0.9145   
       recall            0.9280         0.8896         0.8398         0.7841   

                  water >= 0.80  water >= 0.90  
MEDIAN f1                0.8240         0.7646  
       iou               0.7007         0.6189  
       precision         0.9286         0.9392  
       recall            0.7394         0.6387

In [64]:
fig = go.Figure()

for idx, row in site_medians.iterrows():
    name = idx[1] if isinstance(idx, tuple) else idx

    if name != "iou":
        continue

    x = [float(c[-4:]) for c in site_medians.columns]
    scatter = go.Scatter(x=x, y=row, mode="lines", name=name, line=styles[name]["line"])
    fig.add_trace(scatter)

# Add axis labels
axis_style = {
    "dtick": 0.1,
    "range": [0.05, .95],
    "showline": True,
    "linecolor": "black",
    "mirror": True,
    "showgrid": True,
    "gridcolor": "lightgrey",
    "linewidth": 2,
}

yaxis = axis_style.copy()
yaxis.update({"title": "IoU Score"})
xaxis = axis_style.copy()
xaxis.update({"title": "Water Fraction Threshold"})

fig.update_layout(
    title="Median IoU across Sites",
    plot_bgcolor="white",
    yaxis=yaxis,
    xaxis=xaxis,
)


In [59]:
xaxis

In [49]:
dir(d)

['__class__',
 '__class_getitem__',
 '__contains__',
 '__delattr__',
 '__delitem__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getitem__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__ior__',
 '__iter__',
 '__le__',
 '__len__',
 '__lt__',
 '__ne__',
 '__new__',
 '__or__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__reversed__',
 '__ror__',
 '__setattr__',
 '__setitem__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 'clear',
 'copy',
 'fromkeys',
 'get',
 'items',
 'keys',
 'pop',
 'popitem',
 'setdefault',
 'update',
 'values']

In [51]:
d.update({3: 4})

In [52]:
d

{2: 1, 3: 4}